In [1]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI


In [2]:
load_dotenv()

True

In [3]:
import os
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="openai/gpt-3.5-turbo",   
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    temperature=0
)

In [4]:
# Define graph state 
from typing import TypedDict, List, Dict
from urllib import response

class MeetingNotesState(TypedDict):
    transcript: str
    topics: List[str]
    summary: str
    action_items: List[Dict[str, str]]
    priority: str
    report: str


# input node
def input_node(state: MeetingNotesState):
    return state

# topic extraction agent

def extract_topics(state: MeetingNotesState):
    prompt = f"""
            Extract key topics.

            Return ONLY short phrases.
            No bullets, no numbering, no explanation.

            Transcript:
            {state['transcript']}
             """

    response = llm.invoke(prompt).content
    topics = [line.strip() for line in response.split("\n") if line.strip()]
    new_state = state.copy()
    new_state["topics"] = topics
    return new_state

# summary agent

def summarize_meeting(state: MeetingNotesState):
    prompt = f"""Summarize the following meeting transcript in 3 to 5 sentences.
    transcript: {state['transcript']}"""

    response = llm.invoke(prompt).content

    new_state = state.copy()
    new_state["summary"] = response
    return new_state

# action item extraction agent

def extract_action_items(state: MeetingNotesState):
    prompt = f"""
Extract ALL action items from the meeting.

Definition:
An action item is ANY task, responsibility, or commitment assigned to a person.

Rules:
- Include ALL tasks (even small ones)
- Do NOT miss any
- Format STRICTLY as: task | owner
- One per line
- Use exact role names from transcript (e.g., Mobile Developer, QA Tester)
- If no owner, write: Unassigned
- Do NOT include headings or explanations

Transcript:
{state['transcript']}
"""

    response = llm.invoke(prompt).content

    action_items = []

    for line in response.split("\n"):
        if "|" in line:
            parts = line.split("|")
            if len(parts) == 2:
                task = parts[0].strip()
                owner = parts[1].strip()
                action_items.append({"task": task, "owner": owner})

    new_state = state.copy()
    new_state["action_items"] = action_items
    return new_state

# priority classification agent
def classify_priority(state: MeetingNotesState):
    prompt = f"""Determine priority: High / Medium / Low
    transcript: {state['transcript']}"""

    response = llm.invoke(prompt).content

    new_state = state.copy()
    new_state["priority"] = response.strip()
    return new_state


# conditional logic

def check_action_items(state: MeetingNotesState):
    if len(state["action_items"]) == 0:
        return "no_action"
    else:
        return "has_action"


# final output node

def final_output(state):

    report = ""

    # Title
    report += "Meeting Summary\n"
    report += state["summary"] + "\n\n"

    # Topics (numbered)
    report += "Key Topics\n"
    for i, t in enumerate(state["topics"], 1):
        report += f"{i}. {t}\n"

    report += "\n"

    # Action Items (numbered)
    report += "Action Items\n"
    for i, item in enumerate(state["action_items"], 1):
        report += f"{i}. {item['task']} – {item['owner']}\n"

    report += "\n"

    # Priority
    report += "Priority\n"
    report += state["priority"]

    return {"report": report}




# Build the graph

builder = StateGraph(MeetingNotesState)
builder.add_node("input", input_node)
builder.add_node("extract_topics", extract_topics)
builder.add_node("summarize_meeting", summarize_meeting)
builder.add_node("extract_action_items", extract_action_items)
builder.add_node("classify_priority", classify_priority)
builder.add_node("final_output", final_output)

builder.set_entry_point("input")

builder.add_edge("input", "extract_topics")
builder.add_edge("extract_topics", "summarize_meeting")
builder.add_edge("summarize_meeting", "extract_action_items")
builder.add_conditional_edges("extract_action_items", check_action_items,
                             {"has_action": "classify_priority",
                               "no_action": "final_output"}
                             )
builder.add_edge("classify_priority", "final_output")
builder.add_edge("final_output", END)

graph = builder.compile()




In [5]:
# Step 11: Read Transcript File
# -------------------------------
def load_transcript(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        return file.read().strip()

In [6]:
if __name__ == "__main__":

    transcript = load_transcript("Meeting_Transcript.txt")

    result = graph.invoke({
        "transcript": transcript,
        "topics": [],
        "summary": "",
        "action_items": [],
        "priority": ""
    })

    

In [7]:
print(result.get("report", "No report generated"))

Meeting Summary
The meeting discussed the issue of the mobile app crashing during login, with the team identifying that most of the issues were coming from Android users. They suspected the issue might be related to recent backend changes and decided to investigate both the mobile and backend components. The team agreed to prioritize resolving the issue urgently, with the mobile developer working on a fix and the QA tester preparing a detailed bug report. They also planned to inform users and management about the issue and regroup the next day to review progress.

Key Topics
1. - Mobile app crashing during login
2. - Root cause identification
3. - Android users experiencing crashes
4. - Authentication API issue
5. - Recent backend changes affecting login
6. - Update deployed to authentication service
7. - Investigate mobile and backend components
8. - Urgent resolution needed
9. - Error handling in Android login module
10. - Bug report preparation
11. - Testing on different Android ver